# 02 - Embeddings

Turn CDR3-beta, peptide, and (optionally) the MHC pseudo-sequence into a single
feature matrix via `build_feature_matrix`.

The default backend is the dependency-free **fallback** embedder (k-mer / physico
chemical features) so this runs offline. A one-line switch points at real **ESM-2**
(`esm-hf`) when `transformers` + weights are available.

In [ ]:
from tcr_cliff.data import load_toy
from tcr_cliff.embeddings import build_feature_matrix, embed_pairs, get_embedder
from tcr_cliff.config import EmbeddingConfig

df = load_toy()

# --- OFFLINE (default): fallback embedder ----------------------------------
emb_cfg = EmbeddingConfig(backend='fallback', fallback_dim=64, cache_dir=None)

# --- REAL ESM-2 (one-line switch): uncomment to use HuggingFace ESM-2 -------
# emb_cfg = EmbeddingConfig(backend='esm-hf', model_name='esm2_t12_35M_UR50D')

print('fields embedded:', emb_cfg.fields)
embedder = get_embedder(emb_cfg)
print('embedder name  :', embedder.name)

## Build the concatenated feature matrix

`build_feature_matrix` mean-pools each field and concatenates the per-field blocks.
The `names` list (`"<field>_<j>"`) is what the SHAP attribution in notebook 06 uses
to map importance back onto each sequence source.

In [ ]:
X, names = build_feature_matrix(df, emb_cfg)
print('feature matrix:', X.shape, X.dtype)
print('n feature names:', len(names))
print('first names    :', names[:4], '...')
print('blocks         :', sorted({n.rsplit("_", 1)[0] for n in names}))

## Per-field embeddings and caching

`embed_pairs` returns one `EmbeddingResult` per field. Set `cache_dir` on the config
to persist vectors to disk so overlapping rows/fields are not recomputed (we keep it
`None` here to avoid writing files in the demo).

In [ ]:
embs = embed_pairs(df, emb_cfg)
for field, res in embs.items():
    print(f'{field:11s}: backend={res.backend} dim={res.dim} vectors={res.vectors.shape}')

> **Switching to ESM-2.** With `backend='esm-hf'` the embedder lazily imports
> `transformers`, downloads the named ESM-2 checkpoint once, mean-pools the last
> hidden layer over residues, and caches to `cache_dir`. The downstream API
> (`build_feature_matrix`) is identical - only the config line changes.

### Next
Continue to **03_mine_cliffs** to detect single-residue binding cliffs.